# Task-Based Execution with the Python API

With `openfe quickrun`, you execute a specific `Transformation` at a time. Task-based execution instead takes a whole `AlchemicalNetwork`, breaks it into `tasks`, and lets one or more workers pick up whichever tasks are ready to run. Each task is a single `ProtocolUnit`, for example the setup, simulation or analysis step of one repeat of one `Transformation`.

Three objects do the work:

- the **`Warehouse`** stores everything needed to run the campaign (the network, every task) and every result produced;
- the **`TaskStatusDB`** tracks the status of each task and the dependencies between them;
- a **`Worker`** executes task from the task database.

This tutorial walks through a complete campaign with the Python API. For the equivalent command-line workflow, see the [Task-based Execution page of the OpenFE user guide](https://openfe--2113.org.readthedocs.build/en/2113/guide/execution/task_based_execution.html).

In [1]:
import openfe

## Load an AlchemicalNetwork

The input to a task-based campaign is an `AlchemicalNetwork`. If you don't have one yet, see the [Create an `AlchemicalNetwork` cookbook](https://docs.openfree.energy/en/stable/cookbook/create_alchemical_network.html), or add `--networks-only` to an `openfe plan-rbfe-network` call.

Here we use a small MCL-1 network with pre-charged ligands and two edges, `ligand_1 → ligand_2` and `ligand_2 → ligand_3`. Each edge has a complex and a solvent leg, giving four `Transformations`. 

In [2]:
alchemical_network = openfe.AlchemicalNetwork.from_json("alchemicalNetwork_mc1_small.json")

## Set up the campaign

We use `setup_task_campaign` to create a task database and Warehouse from an `AlchemicalNetwork`. The Warehouse stores the `AlchemicalNetwork` and every `ProtocolUnit` of every `ProtocolDAG` as a task. 
The `TaskStatusDB` containes each task and its dependencies.

This is equivalent to `openfe setup-task-campaign --alchemical-network alchemicalNetwork_mc1_small.json` on the command line.

In [4]:
# build a database of tasks from the AlchemicalNetwork
from openfe.orchestration import setup_task_campaign

campaign_name = "mcl1"
warehouse_dir = f"warehouse_{campaign_name}"
db_path = f"tasks_{campaign_name}.db"

task_db, wh = setup_task_campaign(
    alchemical_network=alchemical_network,
    warehouse_dir=warehouse_dir,
    db_path=db_path
)

In [5]:
task_db

TaskStatusDB(engine=Engine(sqlite:///tasks_mcl1.db))

### The Warehouse

The `Warehouse` is a directory holding several stores: `setup` (the `AlchemicalNetwork` and its individual components), `tasks` (the `ProtocolUnit`s), `protocol_dags` (used to reassemble results later), `results` (every `ProtocolUnitResult`) and `shared` (files passed between units of the same `Transformation`).

In [12]:
!ls warehouse_mcl1/

protocol_dags results       setup         shared        tasks


/Users/hannahbaumann/.local/share/mamba/envs/openfe_dev/lib/python3.13/pty.py:95: DeprecationWarning: This process (pid=8068) is multi-threaded, use of forkpty() may lead to deadlocks in the child.
  pid, fd = os.forkpty()


In [6]:
# the AlchemicalNetwork is now stored in the Warehouse
wh.exists(alchemical_network.key)

True

### The TaskStatusDB

The `TaskStatusDB` is an SQLite database. `get_task_df` returns its task table as a `pandas.DataFrame`, which is the same information shown by `openfe status`.

Each hybrid topology `Transformation` has three units per repeat, so our four `Transformations` give 12 tasks. Only the setup units are `AVAILABLE`; everything else is `BLOCKED` until the tasks it depends on have completed.

In [13]:
from openfe.orchestration import get_task_df, get_dependency_df

task_df = get_task_df(task_db);
task_df 

,taskid,status,last_modified,tries,max_tries
0,HybridTopologySetupUnit-953a4b92b4c44425bb647e...,COMPLETED,2026-09-30 10:48:40.483826,1,1
1,HybridTopologySetupUnit-b5cc67648c434ceca4e69e...,COMPLETED,2026-09-30 10:48:45.666020,1,1
2,HybridTopologySetupUnit-f645288db5164800ab84fa...,COMPLETED,2026-09-30 10:48:46.174758,1,1
3,HybridTopologySetupUnit-eaab94ae3e8b4978a55711...,COMPLETED,2026-09-30 10:48:51.551296,1,1
4,HybridTopologyMultiStateSimulationUnit-d3871d9...,AVAILABLE,2026-09-30 10:48:40.484011,0,1
5,HybridTopologyMultiStateSimulationUnit-19894f4...,AVAILABLE,2026-09-30 10:48:45.666145,0,1
6,HybridTopologyMultiStateSimulationUnit-519c99f...,AVAILABLE,2026-09-30 10:48:46.174914,0,1
7,HybridTopologyMultiStateSimulationUnit-bdee1f0...,AVAILABLE,2026-09-30 10:48:51.551421,0,1
8,HybridTopologyMultiStateAnalysisUnit-aaa8037c1...,BLOCKED,NaT,0,1
9,HybridTopologyMultiStateAnalysisUnit-e193ef5e5...,BLOCKED,NaT,0,1


`get_dependency_df` shows the dependencies between tasks: the task in `from` must complete before the task in `to` can run. Each simulation unit depends on its setup unit, and each analysis unit depends on both.

In [8]:
dep_df = get_dependency_df(task_db);
dep_df

,from,to,blocking
0,HybridTopologySetupUnit-953a4b92b4c44425bb647e...,HybridTopologyMultiStateSimulationUnit-d3871d9...,True
1,HybridTopologySetupUnit-b5cc67648c434ceca4e69e...,HybridTopologyMultiStateSimulationUnit-19894f4...,True
2,HybridTopologySetupUnit-f645288db5164800ab84fa...,HybridTopologyMultiStateSimulationUnit-519c99f...,True
3,HybridTopologySetupUnit-eaab94ae3e8b4978a55711...,HybridTopologyMultiStateSimulationUnit-bdee1f0...,True
4,HybridTopologySetupUnit-953a4b92b4c44425bb647e...,HybridTopologyMultiStateAnalysisUnit-aaa8037c1...,True
5,HybridTopologyMultiStateSimulationUnit-d3871d9...,HybridTopologyMultiStateAnalysisUnit-aaa8037c1...,True
6,HybridTopologySetupUnit-b5cc67648c434ceca4e69e...,HybridTopologyMultiStateAnalysisUnit-e193ef5e5...,True
7,HybridTopologyMultiStateSimulationUnit-19894f4...,HybridTopologyMultiStateAnalysisUnit-e193ef5e5...,True
8,HybridTopologySetupUnit-f645288db5164800ab84fa...,HybridTopologyMultiStateAnalysisUnit-bde91842f...,True
9,HybridTopologyMultiStateSimulationUnit-519c99f...,HybridTopologyMultiStateAnalysisUnit-bde91842f...,True


## Execute tasks with a Worker

A `Worker` needs the `Warehouse` and the path to the `TaskStatusDB`. Each call to `Worker.execute_unit`:

1. checks out the next `AVAILABLE` task, marking it `IN_PROGRESS`;
2. loads the task's `ProtocolUnit`, and the results of any tasks it depends on, from the `Warehouse`;
3. runs the unit, using `scratch` for temporary files;
4. stores the result in the `Warehouse` and marks the task complete, unblocking any tasks that depend on it.

It returns the task ID and its `ProtocolUnitResult`, or `None` if no task is available.

In [9]:
from openfe.orchestration import Worker

worker = Worker(wh, db_path)

Now we run the four setup units (the four `AVAILABLE` tasks):

In [10]:
# run this cell to execute units and show the result of the unit
n_submissions = 4
for _ in range(n_submissions):
    exec_result = worker.execute_unit(scratch='scratch')
    print(exec_result)

INFO:openfe.utils.system_probe.log:SYSTEM CONFIG DETAILS:
INFO:openfe.utils.system_probe.log.hostname:hostname: 'MacBook-Pro-2.local'
INFO:openfe.utils.system_probe.log.gpu:CUDA-based GPU not found
INFO:openfe.utils.system_probe.log:Memory used: 5.4G (80.6%)
INFO:openfe.utils.system_probe.log:scratch/task_workdirs/HybridTopologySetupUnit-953a4b92b4c44425bb647eac913afc5d: 70% full (138.5G free)
INFO:gufekey.openfe.protocols.openmm_rfe.hybridtop_units.HybridTopologySetupUnit:Starting system setup unit
INFO:gufekey.openfe.protocols.openmm_rfe.hybridtop_units.HybridTopologySetupUnit:Parameterizing systems
INFO:openmmforcefields.generators.template_generators:Requested to generate parameters for residue <Residue 0 (LIG) of chain 0>
INFO:openmmforcefields.generators.template_generators:Generating a residue template for [H][c]1[c]([H])[c]([H])[c]2[c]([c]1[H])[S][C]([C](=[O])[O-])=[C]2[Cl] using openff-2.2.1
INFO:openmmforcefields.generators.template_generators:Requested to generate parameters

('HybridTopologySetupUnit-953a4b92b4c44425bb647eac913afc5d', ProtocolUnitResult(HybridTopology Setup: ligand_2 to ligand_3 repeat 0 generation 0))


INFO:openmmforcefields.generators.template_generators:Requested to generate parameters for residue <Residue 153 (LIG) of chain 1>
INFO:openmmforcefields.generators.template_generators:Generating a residue template for [H][c]1[c]([H])[c]([H])[c]2[c]([c]1[H])[S][C]([C](=[O])[O-])=[C]2[Cl] using openff-2.2.1
INFO:openmmforcefields.generators.template_generators:Requested to generate parameters for residue <Residue 8531 (LIG) of chain 4>
INFO:openmmforcefields.generators.template_generators:Generating a residue template for [H][c]1[c]([H])[c]([Cl])[c]2[c]([c]1[H])[S][C]([C](=[O])[O-])=[C]2[Cl] using openff-2.2.1
INFO:openfe.protocols.openmm_rfe._rfe_utils.relative:Creating hybrid system
INFO:openfe.protocols.openmm_rfe._rfe_utils.relative:Setting force field terms
INFO:openfe.protocols.openmm_rfe._rfe_utils.relative:Adding forces
INFO:openfe.protocols.openmm_rfe._rfe_utils.relative:No CMAPTorsionForce found. Skipping adding force.
INFO:openfe.protocols.openmm_rfe._rfe_utils.relative:Hybrid

('HybridTopologySetupUnit-b5cc67648c434ceca4e69e89c195fcc0', ProtocolUnitResult(HybridTopology Setup: ligand_2 to ligand_3 repeat 0 generation 0))


INFO:openmmforcefields.generators.template_generators:Requested to generate parameters for residue <Residue 595 (LIG) of chain 3>
INFO:openmmforcefields.generators.template_generators:Generating a residue template for [H][c]1[c]([H])[c]([H])[c]2[c]([c]1[H])[S][C]([C](=[O])[O-])=[C]2[Cl] using openff-2.2.1
INFO:openfe.protocols.openmm_rfe._rfe_utils.relative:Creating hybrid system
INFO:openfe.protocols.openmm_rfe._rfe_utils.relative:Setting force field terms
INFO:openfe.protocols.openmm_rfe._rfe_utils.relative:Adding forces
INFO:openfe.protocols.openmm_rfe._rfe_utils.relative:No CMAPTorsionForce found. Skipping adding force.
INFO:openfe.protocols.openmm_rfe._rfe_utils.relative:Hybrid system created
/Users/hannahbaumann/.local/share/mamba/envs/openfe_dev/lib/python3.13/site-packages/mdtraj/core/topology.py:97: UserWarning: atom_indices are not monotonically increasing
  warnings.warn("atom_indices are not monotonically increasing")
INFO:exorcist.taskdb:Marking task 'HybridTopologySetupUn

('HybridTopologySetupUnit-f645288db5164800ab84fa85c401f788', ProtocolUnitResult(HybridTopology Setup: ligand_1 to ligand_2 repeat 0 generation 0))


INFO:openmmforcefields.generators.template_generators:Requested to generate parameters for residue <Residue 153 (LIG) of chain 1>
INFO:openmmforcefields.generators.template_generators:Generating a residue template for [H][C]1=[C]([C](=[O])[O-])[S][c]2[c]([H])[c]([H])[c]([H])[c]([H])[c]21 using openff-2.2.1
INFO:openmmforcefields.generators.template_generators:Requested to generate parameters for residue <Residue 8531 (LIG) of chain 4>
INFO:openmmforcefields.generators.template_generators:Generating a residue template for [H][c]1[c]([H])[c]([H])[c]2[c]([c]1[H])[S][C]([C](=[O])[O-])=[C]2[Cl] using openff-2.2.1
INFO:openfe.protocols.openmm_rfe._rfe_utils.relative:Creating hybrid system
INFO:openfe.protocols.openmm_rfe._rfe_utils.relative:Setting force field terms
INFO:openfe.protocols.openmm_rfe._rfe_utils.relative:Adding forces
INFO:openfe.protocols.openmm_rfe._rfe_utils.relative:No CMAPTorsionForce found. Skipping adding force.
INFO:openfe.protocols.openmm_rfe._rfe_utils.relative:Hybrid

('HybridTopologySetupUnit-eaab94ae3e8b4978a55711953dd3a9c2', ProtocolUnitResult(HybridTopology Setup: ligand_1 to ligand_2 repeat 0 generation 0))


Four setup unit are now `COMPLETED`, and the simulation units that depended on them have become `AVAILABLE`:

In [11]:
df=get_task_df(task_db)
df

,taskid,status,last_modified,tries,max_tries
0,HybridTopologySetupUnit-953a4b92b4c44425bb647e...,COMPLETED,2026-09-30 10:48:40.483826,1,1
1,HybridTopologySetupUnit-b5cc67648c434ceca4e69e...,COMPLETED,2026-09-30 10:48:45.666020,1,1
2,HybridTopologySetupUnit-f645288db5164800ab84fa...,COMPLETED,2026-09-30 10:48:46.174758,1,1
3,HybridTopologySetupUnit-eaab94ae3e8b4978a55711...,COMPLETED,2026-09-30 10:48:51.551296,1,1
4,HybridTopologyMultiStateSimulationUnit-d3871d9...,AVAILABLE,2026-09-30 10:48:40.484011,0,1
5,HybridTopologyMultiStateSimulationUnit-19894f4...,AVAILABLE,2026-09-30 10:48:45.666145,0,1
6,HybridTopologyMultiStateSimulationUnit-519c99f...,AVAILABLE,2026-09-30 10:48:46.174914,0,1
7,HybridTopologyMultiStateSimulationUnit-bdee1f0...,AVAILABLE,2026-09-30 10:48:51.551421,0,1
8,HybridTopologyMultiStateAnalysisUnit-aaa8037c1...,BLOCKED,NaT,0,1
9,HybridTopologyMultiStateAnalysisUnit-e193ef5e5...,BLOCKED,NaT,0,1


In [38]:
while True:
    execution = worker.execute_unit(scratch="scratch")
    if execution is None:
        break
    taskid, result = execution
    print(f"{taskid} succeeded: {result.ok()}")

INFO:exorcist.taskdb:Checking out a new task
INFO:exorcist.taskdb:Selected task 'HybridTopologyMultiStateSimulationUnit-519c99f8253d472d94b7d1925c385685'
INFO:gufekey.gufe.components.smallmoleculecomponent.SmallMoleculeComponent:Partial charges are present for SmallMoleculeComponent-e5a4596e9d800367c6832564a93877f3 (name: 'ligand_1')
INFO:gufekey.gufe.components.smallmoleculecomponent.SmallMoleculeComponent:Partial charges are present for SmallMoleculeComponent-e5a4596e9d800367c6832564a93877f3 (name: 'ligand_1')
INFO:gufekey.gufe.components.smallmoleculecomponent.SmallMoleculeComponent:Partial charges are present for SmallMoleculeComponent-8f5b6a9848f63293c9b2361b51b466ab (name: 'ligand_2')
INFO:gufekey.gufe.components.smallmoleculecomponent.SmallMoleculeComponent:Partial charges are present for SmallMoleculeComponent-e5a4596e9d800367c6832564a93877f3 (name: 'ligand_1')
INFO:gufekey.gufe.components.smallmoleculecomponent.SmallMoleculeComponent:Partial charges are present for SmallMolecu

Please cite the following:

        Friedrichs MS, Eastman P, Vaidyanathan V, Houston M, LeGrand S, Beberg AL, Ensign DL, Bruns CM, and Pande VS. Accelerating molecular dynamic simulations on graphics processing unit. J. Comput. Chem. 30:864, 2009. DOI: 10.1002/jcc.21209
        Eastman P and Pande VS. OpenMM: A hardware-independent framework for molecular simulations. Comput. Sci. Eng. 12:34, 2010. DOI: 10.1109/MCSE.2010.27
        Eastman P and Pande VS. Efficient nonbonded interactions for molecular dynamics on a graphics processing unit. J. Comput. Chem. 31:1268, 2010. DOI: 10.1002/jcc.21413
        Eastman P and Pande VS. Constant constraint matrix approximation: A robust, parallelizable constraint method for molecular simulations. J. Chem. Theor. Comput. 6:434, 2010. DOI: 10.1021/ct900463w
        Chodera JD and Shirts MR. Replica exchange and expanded ensemble simulations as Gibbs multistate: Simple improvements for enhanced mixing. J. Chem. Phys., 135:194110, 2011. DOI:10.1063/

INFO:gufekey.openfe.protocols.openmm_rfe.hybridtop_units.HybridTopologyMultiStateSimulationUnit:equilibrating systems
INFO:openmmtools.multistate.multistatesampler:Equilibration iteration 1/5
/Users/hannahbaumann/.local/share/mamba/envs/openfe_dev/lib/python3.13/site-packages/openmmtools/mcmc.py:723: UserWarning: Particle coordinate is NaN.  For more information, see https://github.com/openmm/openmm/wiki/Frequently-Asked-Questions#nan
  warnings.warn(str(e))
ERROR:openmmtools.mcmc:Potential energy is NaN after 19 attempts of integration with move LangevinDynamicsMove Trying to reinitialize Context as a last-resort restart attempt...
ERROR:openmmtools.mcmc:Potential energy is NaN after 20 attempts of integration with move LangevinDynamicsMove
CRITICAL:openmmtools.multistate.multistatesampler:Propagating replica 0 at state 0 resulted in a NaN!
The state of the system and integrator before the error were saved in /Users/hannahbaumann/ExampleNotebooks/execution/warehouse_mcl1/shared/task_w

HybridTopologyMultiStateSimulationUnit-519c99f8253d472d94b7d1925c385685 succeeded: False


INFO:openfe.utils.system_probe.log:SYSTEM CONFIG DETAILS:
INFO:openfe.utils.system_probe.log.hostname:hostname: 'MacBook-Pro-2.local'
INFO:openfe.utils.system_probe.log.gpu:CUDA-based GPU not found
INFO:openfe.utils.system_probe.log:Memory used: 5.3G (81.8%)
INFO:openfe.utils.system_probe.log:scratch/task_workdirs/HybridTopologyMultiStateSimulationUnit-bdee1f088c91432da7a950b04f6aa955: 70% full (136.7G free)
INFO:gufekey.openfe.protocols.openmm_rfe.hybridtop_units.HybridTopologyMultiStateSimulationUnit:Starting simulation unit
/Users/hannahbaumann/.local/share/mamba/envs/openfe_dev/lib/python3.13/site-packages/openfe/protocols/openmm_utils/omm_compute.py:78: UserWarning: Non-CUDA platform selected: CPU, this may significantly impact simulation performance
  warnings.warn(wmsg)


Please cite the following:

        Friedrichs MS, Eastman P, Vaidyanathan V, Houston M, LeGrand S, Beberg AL, Ensign DL, Bruns CM, and Pande VS. Accelerating molecular dynamic simulations on graphics processing unit. J. Comput. Chem. 30:864, 2009. DOI: 10.1002/jcc.21209
        Eastman P and Pande VS. OpenMM: A hardware-independent framework for molecular simulations. Comput. Sci. Eng. 12:34, 2010. DOI: 10.1109/MCSE.2010.27
        Eastman P and Pande VS. Efficient nonbonded interactions for molecular dynamics on a graphics processing unit. J. Comput. Chem. 31:1268, 2010. DOI: 10.1002/jcc.21413
        Eastman P and Pande VS. Constant constraint matrix approximation: A robust, parallelizable constraint method for molecular simulations. J. Chem. Theor. Comput. 6:434, 2010. DOI: 10.1021/ct900463w
        Chodera JD and Shirts MR. Replica exchange and expanded ensemble simulations as Gibbs multistate: Simple improvements for enhanced mixing. J. Chem. Phys., 135:194110, 2011. DOI:10.1063/

INFO:gufekey.openfe.protocols.openmm_rfe.hybridtop_units.HybridTopologyMultiStateSimulationUnit:minimizing systems
INFO:gufekey.openfe.protocols.openmm_rfe.hybridtop_units.HybridTopologyMultiStateSimulationUnit:equilibrating systems
INFO:openmmtools.multistate.multistatesampler:Equilibration iteration 1/5
/Users/hannahbaumann/.local/share/mamba/envs/openfe_dev/lib/python3.13/site-packages/openmmtools/mcmc.py:723: UserWarning: Particle coordinate is NaN.  For more information, see https://github.com/openmm/openmm/wiki/Frequently-Asked-Questions#nan
  warnings.warn(str(e))
ERROR:openmmtools.mcmc:Potential energy is NaN after 19 attempts of integration with move LangevinDynamicsMove Trying to reinitialize Context as a last-resort restart attempt...
ERROR:openmmtools.mcmc:Potential energy is NaN after 20 attempts of integration with move LangevinDynamicsMove
CRITICAL:openmmtools.multistate.multistatesampler:Propagating replica 1 at state 1 resulted in a NaN!
The state of the system and int

HybridTopologyMultiStateSimulationUnit-bdee1f088c91432da7a950b04f6aa955 succeeded: False


In [39]:
df=get_task_df(task_db)
df

,taskid,status,last_modified,tries,max_tries
0,HybridTopologySetupUnit-953a4b92b4c44425bb647e...,COMPLETED,2026-09-30 10:48:40.483826,1,1
1,HybridTopologySetupUnit-b5cc67648c434ceca4e69e...,COMPLETED,2026-09-30 10:48:45.666020,1,1
2,HybridTopologySetupUnit-f645288db5164800ab84fa...,COMPLETED,2026-09-30 10:48:46.174758,1,1
3,HybridTopologySetupUnit-eaab94ae3e8b4978a55711...,COMPLETED,2026-09-30 10:48:51.551296,1,1
4,HybridTopologyMultiStateSimulationUnit-d3871d9...,COMPLETED,2026-09-30 11:18:06.728680,1,1
5,HybridTopologyMultiStateSimulationUnit-19894f4...,COMPLETED,2026-09-30 11:22:18.853337,1,1
6,HybridTopologyMultiStateSimulationUnit-519c99f...,TOO_MANY_RETRIES,2026-09-30 14:52:02.582027,5,5
7,HybridTopologyMultiStateSimulationUnit-bdee1f0...,TOO_MANY_RETRIES,2026-09-30 14:53:08.231210,5,5
8,HybridTopologyMultiStateAnalysisUnit-aaa8037c1...,COMPLETED,2026-09-30 11:24:05.750435,1,1
9,HybridTopologyMultiStateAnalysisUnit-e193ef5e5...,COMPLETED,2026-09-30 11:24:26.256369,1,1


## Gather results

The `Warehouse` now contains a `ProtocolUnitResult` for every completed task. `gather_all_results` reassembles these into one `(ProtocolResult, ProtocolDAGResult)` pair per `ProtocolDAG`.

Because each `ProtocolDAG` corresponds to one `Transformation`, these are per-leg free energies (the complex or the solvent leg of an edge), not relative binding free energies. See below for combining them.

You can load an existing `Warehouse` from any session with `FileSystemWarehouse.from_dir`:

**TODO: rerun to get results with 3 repeats per edge, upload to zenodo**

In [31]:
# load a warehouse
from openfe.storage.warehouse import FileSystemWarehouse

results_wh = FileSystemWarehouse.from_dir(warehouse_dir)
result_edges = results_wh.gather_all_results()

INFO:gufekey.gufe.components.smallmoleculecomponent.SmallMoleculeComponent:Partial charges are present for SmallMoleculeComponent-e42f2d6e8715f91ce1dd6dd3a3b62fe9 (name: 'ligand_3')
INFO:gufekey.gufe.components.smallmoleculecomponent.SmallMoleculeComponent:Partial charges are present for SmallMoleculeComponent-8f5b6a9848f63293c9b2361b51b466ab (name: 'ligand_2')
INFO:gufekey.gufe.components.smallmoleculecomponent.SmallMoleculeComponent:Partial charges are present for SmallMoleculeComponent-e42f2d6e8715f91ce1dd6dd3a3b62fe9 (name: 'ligand_3')
INFO:gufekey.gufe.components.smallmoleculecomponent.SmallMoleculeComponent:Partial charges are present for SmallMoleculeComponent-8f5b6a9848f63293c9b2361b51b466ab (name: 'ligand_2')
INFO:gufekey.gufe.components.smallmoleculecomponent.SmallMoleculeComponent:Partial charges are present for SmallMoleculeComponent-8f5b6a9848f63293c9b2361b51b466ab (name: 'ligand_2')
INFO:gufekey.gufe.components.smallmoleculecomponent.SmallMoleculeComponent:Partial charges

In [32]:
result_edges = results_wh.gather_all_results();

INFO:gufekey.gufe.components.smallmoleculecomponent.SmallMoleculeComponent:Partial charges are present for SmallMoleculeComponent-e42f2d6e8715f91ce1dd6dd3a3b62fe9 (name: 'ligand_3')
INFO:gufekey.gufe.components.smallmoleculecomponent.SmallMoleculeComponent:Partial charges are present for SmallMoleculeComponent-8f5b6a9848f63293c9b2361b51b466ab (name: 'ligand_2')
INFO:gufekey.gufe.components.smallmoleculecomponent.SmallMoleculeComponent:Partial charges are present for SmallMoleculeComponent-e42f2d6e8715f91ce1dd6dd3a3b62fe9 (name: 'ligand_3')
INFO:gufekey.gufe.components.smallmoleculecomponent.SmallMoleculeComponent:Partial charges are present for SmallMoleculeComponent-8f5b6a9848f63293c9b2361b51b466ab (name: 'ligand_2')
INFO:gufekey.gufe.components.smallmoleculecomponent.SmallMoleculeComponent:Partial charges are present for SmallMoleculeComponent-8f5b6a9848f63293c9b2361b51b466ab (name: 'ligand_2')
INFO:gufekey.gufe.components.smallmoleculecomponent.SmallMoleculeComponent:Partial charges

In [33]:
result_edges

[(<RelativeHybridTopologyProtocolResult-6ce1a0703a8b10de1fbd858861957581>,
  <ProtocolDAGResult-00ec68fe52deaad40e74e9a2d7fc4df8>),
 (<RelativeHybridTopologyProtocolResult-8301d203efc373156a71d2307c2b45b4>,
  <ProtocolDAGResult-db095c6629000d7f4bfe12606cae6d80>),
 (<RelativeHybridTopologyProtocolResult-bc85ae4153ed5d9a2e9db1ac1c2e5e51>,
  <ProtocolDAGResult-e46088ed77bbde2bd16dfc157ba7f749>),
 (<RelativeHybridTopologyProtocolResult-8301d203efc373156a71d2307c2b45b4>,
  <ProtocolDAGResult-6239994c90c3cd52e1a660b6519c9309>)]

In [18]:
# [(pr.get_estimate(), pr.get_uncertainty()) for pr, dag in result_edges if dag.ok()]